[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/solutions/b_02_vmap_batching_solution.ipynb)

# 🟢 Solution: Pairwise Distances with vmap

*JAX Fundamentals · Easy*

Reference implementation. Try it yourself in `b_02_vmap_batching.ipynb` first.

---
Given `X` of shape `(N, D)` and `Y` of shape `(M, D)`, compute the matrix of
**squared Euclidean distances** of shape `(N, M)`:

$$D_{ij} = \|x_i - y_j\|_2^2 = \sum_{d} (X_{id} - Y_{jd})^2$$

The point of this problem is *not* the math — it is learning to write the
function for a **single example** and let `jax.vmap` add the batch dimensions.
This "write one, vmap the rest" habit is what interviewers are looking for.

### Rules
- Write a single-pair helper, then compose **two** `vmap`s around it
- No Python `for` loops over N or M
- Do **not** use the expand-dims broadcasting trick (`X[:, None] - Y[None]`);
  the exercise is specifically about `vmap` and `in_axes`
- Do not use `jnp.linalg.norm` or `scipy` distance helpers

### Example
```
X shape (3, 2), Y shape (5, 2)  ->  output shape (3, 5)
```

### Why it matters
The expand-dims version materializes an `(N, M, D)` intermediate. `vmap` expresses
the same computation without you hand-managing axes, and it composes with `grad`
and `jit`. Being fluent with `in_axes=(None, 0)` vs `(0, None)` is a very common
JAX screening question.

In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge flax')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✅ REFERENCE SOLUTION

import jax
import jax.numpy as jnp


def pairwise_sq_dist(X, Y):
    # The whole problem, for one pair of vectors.
    def sq_dist(x, y):
        diff = x - y
        return jnp.sum(diff * diff)

    # One row: fix x, sweep over every y.
    row = jax.vmap(sq_dist, in_axes=(None, 0))
    # Full matrix: sweep that row-builder over every x.
    return jax.vmap(row, in_axes=(0, None))(X, Y)

In [ ]:
# 🔍 Verify
import jax.numpy as jnp

X = jnp.array([[0.0, 0.0], [1.0, 0.0], [0.0, 1.0]])
Y = jnp.array([[0.0, 0.0], [1.0, 1.0]])

out = pairwise_sq_dist(X, Y)
print("X:", X.shape, " Y:", Y.shape)
print("out shape:", out.shape, "(expected (3, 2))")
print(out)
# Row 0 is distance from origin to each Y -> [0., 2.]

In [ ]:
# Run the judge against the reference solution
from jax_judge import check

check("vmap_batching")